In [1]:
# Get daily constraint ranked by the abs RT - DA 
import sys
sys.path.append('/var/www/python/Prod/nighthawk/')

import pandas as pd
from nighthawk.data import Constraint

In [2]:
now = pd.Timestamp.now(tz='US/Central')
days_ahead = 2 if now.hour >= 10 else 1
bid_dt = (now + pd.Timedelta(days=days_ahead)).strftime('%Y-%m-%d')
print(bid_dt)

2026-05-30


# Daily RT DA Spike Analysis 

In [3]:
# give a date and returns to me the hourly metrics at that hour, wind/load/temperature/gas/wind ramp/genoutage

In [4]:
from nighthawk.data.pipeline.common_functions.wind import Wind
from nighthawk.data.pipeline.common_functions.load import Load
from nighthawk.data.pipeline.common_functions.gas import Gas
from nighthawk.data.pipeline.common_functions.genoutage import GenOutage
from nighthawk.data.pipeline.common_functions.weather import Weather
from nighthawk.data.network.node import Node

SPP_HUB_NODES = {636:'south_hub'}
SPP_CITIES =[ ('Kansas City', 'MO'), ('Oklahoma City', 'OK')]


def get_hourly_snapshot(date: str, hour: int):
    assert 1 <= hour <= 24, "hour must be between 1 and 24"
    dt      = date
    dt_prev = (pd.Timestamp(dt) - pd.Timedelta(days=1)).strftime('%Y-%m-%d')

    wind_df = Wind('SPP').get_total_wind(dt_prev, dt, var_spec=['f'])
    load_df = Load('SPP').get_total_load(dt_prev, dt, var_spec=['f'])

    gas_raw = Gas('SPP').get_daily_gas_price(['Henry'], dt_prev, dt, pivot=False)
    gas_df  = (gas_raw[gas_raw['hub_name'] == 'Henry'][['dt', 'gas_price']]
               .rename(columns={'gas_price': 'henry_gas_price'}))

    go_raw = GenOutage('SPP').get_genoutage_by_level(dt_prev, dt, var_spec=['f'], area_list=['SPP'])
    go_df  = go_raw[go_raw['baa_zone'] == 'SPP'][['dt', 'hr', 'spp_genoutage_forecast_f']]

    weather_obj = Weather('SPP')
    city_ids    = weather_obj.get_city_details(SPP_CITIES)['CityId'].tolist()
    temp_raw    = weather_obj.get_citylevel_temperature_for_ve(dt_prev, dt, city_ids, pivot=False)
    temp_df     = (temp_raw.groupby(['dt', 'hr'])['temperature_degf']
                           .mean().reset_index()
                           .rename(columns={'temperature_degf': 'avg_temp_f'}))

    price_raw = Node(list(SPP_HUB_NODES.keys()), 'SPP').get_price(
        dt, dt, component=['Slack'], type=['DA', 'RT'], granularity='hourly'
    )
    price_raw['dt'] = price_raw['dt'].astype(str)
    price_raw['hr'] = price_raw['hr'].astype(int)

    for df in [wind_df, load_df, go_df, temp_df]:
        df['dt'] = df['dt'].astype(str)
        df['hr'] = df['hr'].astype(int)
    gas_df['dt'] = gas_df['dt'].astype(str)

    base = (
        wind_df[['dt', 'hr', 'spp_wind_total_forecast_f']]
        .merge(load_df[['dt', 'hr', 'spp_load_total_forecast_f']], on=['dt', 'hr'], how='outer')
        .merge(go_df,   on=['dt', 'hr'], how='left')
        .merge(temp_df, on=['dt', 'hr'], how='left')
        .merge(gas_df,  on='dt',         how='left')
        .sort_values(['dt', 'hr']).reset_index(drop=True)
    )
    base['B_wind_ramp'] = base['spp_wind_total_forecast_f'].diff()
    base['B_load_ramp'] = base['spp_load_total_forecast_f'].diff()
    base['B_wind_ramp_2'] = base['spp_wind_total_forecast_f'].diff(2)
    base['B_load_ramp_2'] = base['spp_load_total_forecast_f'].diff(2)

    row       = base[(base['dt'] == dt) & (base['hr'] == hour)]
    price_row = price_raw[(price_raw['dt'] == dt) & (price_raw['hr'] == hour)].copy()
    price_row['hub'] = price_row['node_num'].map(SPP_HUB_NODES)

    if row.empty:
        print(f"No data found for {dt} hour {hour}")
        return None

    r   = row.iloc[0]
    rec = {
        'dt':               dt,
        'hr':               hour,
        'wind_f (MW)':      round(r['spp_wind_total_forecast_f'], 1),
        'load_f (MW)':      round(r['spp_load_total_forecast_f'], 1),
        'genoutage_f (MW)': round(r['spp_genoutage_forecast_f'],  1),
        'avg_temp (°F)':    round(r['avg_temp_f'],                1),
        'henry_gas ($/MMBtu)': round(r['henry_gas_price'],        3),
        'wind_ramp (MW/hr)': round(r['B_wind_ramp'],                1),
        'load_ramp (MW/hr)': round(r['B_load_ramp'],                1),
        'wind_ramp_2 (MW/hr)': round(r['B_wind_ramp_2'],                1),
        'load_ramp_2 (MW/hr)': round(r['B_load_ramp_2'],                1),
    }

    for _, pr in price_row.sort_values('node_num').iterrows():
        hub = pr['hub']
        rec[f'{hub}_da_slack']  = round(pr.get('da_slack', float('nan')), 2)
        rec[f'{hub}_rt_slack']  = round(pr.get('rt_slack', float('nan')), 2)

    display(pd.DataFrame([rec]))
    return pd.DataFrame([rec])


# ── Example ───────────────────────────────────────────────
get_hourly_snapshot('2022-12-23', 18)


/tmp/ipykernel_3362251/756036728.py:28: DeprecationWarning: Call to deprecated function get_city_details.
  city_ids    = weather_obj.get_city_details(SPP_CITIES)['CityId'].tolist()


,dt,hr,wind_f (MW),load_f (MW),genoutage_f (MW),avg_temp (°F),henry_gas ($/MMBtu),wind_ramp (MW/hr),load_ramp (MW/hr),wind_ramp_2 (MW/hr),load_ramp_2 (MW/hr),south_hub_da_slack,south_hub_rt_slack
0,2022-12-23,18,10722.4,40623.0,11765.4,10.3,7.28,-930.1,1710.0,-2080.7,2397.0,149.7,1395.17


,dt,hr,wind_f (MW),load_f (MW),genoutage_f (MW),avg_temp (°F),henry_gas ($/MMBtu),wind_ramp (MW/hr),load_ramp (MW/hr),wind_ramp_2 (MW/hr),load_ramp_2 (MW/hr),south_hub_da_slack,south_hub_rt_slack
0,2022-12-23,18,10722.4,40623.0,11765.4,10.3,7.28,-930.1,1710.0,-2080.7,2397.0,149.7,1395.17


In [5]:
import sys
sys.path.append('/var/www/python/Prod/nighthawk/')
import pandas as pd
from nighthawk.util.bigquery_functions import download_df_from_bq

PNL_COLS = ['clear_mw', 'profit_total', 'profit_congestion', 'profit_slack']

def _fetch_pnl(start_dt: str, end_dt: str) -> pd.DataFrame:
    query = f"""
        SELECT dt, hr, incdec, strategy, rep_zone, broad_zone,
               SUM(clear_mw)          AS clear_mw,
               SUM(profit_total)      AS profit_total,
               SUM(profit_congestion) AS profit_congestion,
               SUM(profit_slack)      AS profit_slack
        FROM `movetocloud-999.virtual_financials.segment_portfolio_details_SPP`
        WHERE dt BETWEEN '{start_dt}' AND '{end_dt}'
        GROUP BY dt, hr, incdec, strategy, rep_zone, broad_zone
        ORDER BY dt, hr
    """
    df = download_df_from_bq(query)
    df['dt'] = pd.to_datetime(df['dt']).dt.strftime('%Y-%m-%d')
    df['hr'] = df['hr'].astype(int)
    return df


def get_spp_pnl(start_dt: str, end_dt: str, group_by: str = 'daily') -> pd.DataFrame:
    """
    Fetch SPP virtual portfolio PnL summed across all strategies.

    group_by: 'daily'  — one row per dt
              'hourly' — one row per dt x hr
              'raw'    — full detail (strategy / rep_zone / incdec)
    """
    assert group_by in ('daily', 'hourly', 'raw')
    df = _fetch_pnl(start_dt, end_dt)
    if group_by == 'daily':
        return df.groupby('dt', as_index=False)[PNL_COLS].sum()
    elif group_by == 'hourly':
        return df.groupby(['dt', 'hr'], as_index=False)[PNL_COLS].sum()
    return df


def get_pnl_snapshot(date: str, hour: int) -> pd.DataFrame:
    """Return a single-row DataFrame with total PnL for one specific date and hour."""
    df = _fetch_pnl(date, date)
    row = df[df['hr'] == hour][PNL_COLS].sum()
    result = pd.DataFrame([{'dt': date, 'hr': hour, **{c: round(row[c], 2) for c in PNL_COLS}}])
    display(result)
    return result


# Daily PnL (summed across all strategies, one row per dt)
daily = get_spp_pnl('2026-05-01', '2026-05-12', group_by='daily')
display(daily)

# Hourly PnL (one row per dt x hr)
hourly = get_spp_pnl('2026-05-01', '2026-05-12', group_by='hourly')
display(hourly)

# Single dt + hour snapshot
get_pnl_snapshot('2022-12-23', 18)


,dt,clear_mw,profit_total,profit_congestion,profit_slack
0,2026-05-01,1314.123994,10346.707933,-1462.969181,10884.276826
1,2026-05-02,1652.635993,-7083.557217,-12633.139192,21.284947
2,2026-05-03,3990.475993,7572.335125,-11842.373297,16723.520420
3,2026-05-04,2913.842014,-3192.139128,-4567.698330,114.950350
4,2026-05-05,4434.162016,16118.681327,8296.839209,4425.614313
5,2026-05-06,3987.468992,-6386.306198,-23104.017113,-674.526433
6,2026-05-07,3755.137000,15738.199106,19953.350274,-7964.616577
7,2026-05-08,4304.602005,67259.501733,29093.770899,32013.608437
8,2026-05-09,3716.091997,16205.688033,1858.715909,16642.058252
9,2026-05-10,4435.628995,3100.265405,2119.289315,206.546872


,dt,hr,clear_mw,profit_total,profit_congestion,profit_slack
0,2026-05-01,1,0.000000,0.000000,0.000000,0.000000
1,2026-05-01,2,159.647999,516.767106,1.068651,278.250245
2,2026-05-01,3,155.144999,466.455470,3.612927,239.142706
3,2026-05-01,4,192.359999,508.910930,14.164024,324.564199
4,2026-05-01,5,55.928000,88.684027,1.266369,90.660056
...,...,...,...,...,...,...
283,2026-05-12,20,303.584000,4922.006249,-1673.332169,6052.511373
284,2026-05-12,21,139.781999,222.990298,-1267.729583,1051.766621
285,2026-05-12,22,178.310000,-2155.851849,-1877.787045,-597.293858
286,2026-05-12,23,148.264000,-763.514208,-667.330491,-334.509082


,dt,hr,clear_mw,profit_total,profit_congestion,profit_slack
0,2022-12-23,18,7.09,5328.41,493.17,4666.42


,dt,hr,clear_mw,profit_total,profit_congestion,profit_slack
0,2022-12-23,18,7.09,5328.41,493.17,4666.42


In [6]:
import os
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

SAVE_PATH   = '/mnt/disks/filedisk1/SPP/VE/spp_hourly_fundamentals.csv'
RF_FEATURES = [
    'wind_f (MW)', 'load_f (MW)', 'genoutage_f (MW)', 'avg_temp (°F)',
    'henry_gas ($/MMBtu)', 'wind_ramp (MW/hr)', 'load_ramp (MW/hr)',
    'wind_ramp_2 (MW/hr)', 'load_ramp_2 (MW/hr)',
]
TARGET = 'SHub_rt_slack'


def find_similar_hours(date: str, hour: int, k: int = 20,
                        dataset_path: str = SAVE_PATH,
                        n_estimators: int = 200) -> pd.DataFrame:
    """
    Train a RandomForest on (fundamentals -> SHub_rt_slack) using all history
    strictly before the given dt/hr, then rank historical hours by RF proximity
    (fraction of trees where a historical row shares the same leaf as the query).

    Returns top-k most similar rows sorted by rf_proximity descending,
    with the query row prepended (rf_proximity = 1.0).
    """
    df = pd.read_csv(dataset_path)
    df['dt'] = df['dt'].astype(str)
    df['hr'] = df['hr'].astype(int)

    # --- strict past-only filter ---
    cutoff    = pd.Timestamp(date) + pd.Timedelta(hours=hour - 1)
    df['_ts'] = pd.to_datetime(df['dt']) + pd.to_timedelta(df['hr'] - 1, unit='h')
    hist      = df[df['_ts'] < cutoff].drop(columns='_ts').reset_index(drop=True)

    # --- query row ---
    query_rows = df[(df['dt'] == date) & (df['hr'] == hour)].drop(columns='_ts', errors='ignore')
    if query_rows.empty:
        print('Query dt/hr not in dataset, fetching live...')
        query_row = get_hourly_snapshot(date, hour)
    else:
        query_row = query_rows.iloc[[0]]

    # --- feature matrix ---
    feat_cols = [c for c in RF_FEATURES if c in hist.columns and c in query_row.columns]
    train_mask = hist[feat_cols].notna().all(axis=1) & hist[TARGET].notna()
    hist_clean = hist[train_mask].reset_index(drop=True)

    X_train = hist_clean[feat_cols].values
    y_train = hist_clean[TARGET].values
    X_query = query_row[feat_cols].fillna(0).values

    # --- train RF ---
    rf = RandomForestRegressor(n_estimators=n_estimators, random_state=42,
                               n_jobs=-1, max_features='sqrt')
    rf.fit(X_train, y_train)

    print(f'RF trained on {len(X_train)} rows | '
          f'top features: {sorted(zip(rf.feature_importances_, feat_cols), reverse=True)[:3]}')

    # --- RF proximity: fraction of trees sharing the same leaf ---
    # apply() returns shape (n_samples, n_estimators) — leaf index per tree
    hist_leaves  = rf.apply(X_train)          # (n_hist, n_trees)
    query_leaves = rf.apply(X_query)          # (1,      n_trees)
    proximity    = (hist_leaves == query_leaves).mean(axis=1)  # (n_hist,)

    # --- top k by proximity ---
    k = min(k, len(hist_clean))
    top_idx  = np.argsort(proximity)[::-1][:k]
    neighbours = hist_clean.iloc[top_idx].copy()
    neighbours.insert(0, 'rf_proximity', proximity[top_idx].round(4))
    neighbours = neighbours.sort_values('rf_proximity', ascending=False).reset_index(drop=True)

    # --- remove any row from the query date before prepending query row ---
    neighbours = neighbours[neighbours['dt'] != date].sort_values('rf_proximity',ascending=False)
    # neighbours = neighbours.sort_values('SHub_rt_slack', ascending=False).groupby('dt').head(3).sort_values('SHub_rt_slack', ascending=False)
    # --- prepend query row ---
    q = query_row.copy()
    q.insert(0, 'rf_proximity', 1.0)
    result = pd.concat([q, neighbours], ignore_index=True)
    return result


# ── Example ───────────────────────────────────────────────
find_similar_hours('2026-02-05', 17, k=20)


RF trained on 53458 rows | top features: [(np.float64(0.2462259813613005), 'avg_temp (°F)'), (np.float64(0.15052681429048678), 'genoutage_f (MW)'), (np.float64(0.1383914165600613), 'henry_gas ($/MMBtu)')]


,rf_proximity,dt,hr,wind_f (MW),load_f (MW),genoutage_f (MW),avg_temp (°F),henry_gas ($/MMBtu),wind_ramp (MW/hr),load_ramp (MW/hr),wind_ramp_2 (MW/hr),load_ramp_2 (MW/hr),SHub_da_slack,SHub_rt_slack,clear_mw,profit_total,profit_congestion,profit_slack
0,1.000,2026-02-05,17,14289.63,32834.0,10650.1,59.500000,6.43,-1461.28,295.0,-2410.89,11.0,28.1054,404.1851,206.440001,-62365.938848,387.063636,-65353.675389
1,0.170,2022-07-01,6,14438.65,30901.0,8863.1,73.166667,6.46,-1117.83,138.0,-2486.89,-172.0,30.8206,45.4116,23.939000,-40.729554,74.293929,-263.847374
2,0.075,2022-08-16,6,13548.89,32258.0,6419.1,74.166667,8.55,-1221.32,342.0,-2240.43,31.0,40.2045,55.4238,103.304000,957.512408,908.283454,-579.449514
3,0.045,2022-07-25,6,14579.86,33429.0,7723.6,75.833333,8.24,-1078.63,420.0,-2080.29,51.0,36.4565,49.6445,61.907000,-192.575932,-491.847419,-21.042240
4,0.045,2022-04-13,19,14265.99,29406.0,21813.5,52.333333,6.56,-1165.66,-29.0,-2153.31,-81.0,50.7048,44.5023,109.996001,1498.184027,1851.563093,-600.867565
5,0.045,2022-09-29,10,14386.65,28338.0,16650.0,59.000000,6.60,-1142.15,482.0,-1888.43,831.0,40.2965,62.2358,156.316001,244.029452,3065.127021,-3467.083809
6,0.040,2022-04-25,12,14301.43,28245.0,24450.6,52.333333,6.55,-1507.18,-42.0,-2526.24,126.0,41.8679,24.2558,117.336000,5086.091929,5225.552505,-458.295572
7,0.040,2022-12-19,12,13647.13,34908.0,14665.5,40.500000,6.59,-1428.13,-398.0,-3147.18,-656.0,62.9655,55.5540,54.907000,259.616920,177.717612,-65.472462
8,0.040,2022-06-18,8,13801.18,30728.0,9573.1,75.500000,7.34,-1470.64,1218.0,-2445.80,1102.0,39.2255,47.2523,39.387000,893.380030,880.272826,-203.056099
9,0.035,2022-07-01,5,15556.48,30763.0,8863.1,73.500000,6.46,-1369.06,-310.0,-2613.74,-958.0,23.0436,47.1452,22.901000,-197.759024,-1.283994,-348.660561


In [7]:
dt_hr_list = [(bid_dt, hr) for hr in range(1, 25)]

all_results = {}
avg_rt_slack_list = []
dangerous_hours = []
avg_da_slack_list=[]

for dt, hr in dt_hr_list:
    print(f'\n=== {dt} hr {hr} ===')
    result = find_similar_hours(dt, hr, k=20)
    all_results[(dt, hr)] = result
    display(result[:5])

    neighbours = result[result['dt'] != dt]
    avg_slack = neighbours['SHub_rt_slack'].mean()
    avg_rt_slack_list.append({'dt': dt, 'hr': hr, 'avg_rt_slack': round(avg_slack, 2)})
    avg_slack = neighbours['SHub_da_slack'].mean()
    avg_da_slack_list.append({'dt': dt, 'hr': hr, 'avg_da_slack': round(avg_slack, 2)})
    

    if (neighbours['SHub_rt_slack'] > 150).any():
        dangerous_hours.append({'dt': dt, 'hr': hr, 'avg_rt_slack': round(avg_slack, 2)})

print('\n=== Avg RT Slack by Hour ===')
display(pd.DataFrame(avg_rt_slack_list))
print('\n=== Avg DA Slack by Hour ===')
display(pd.DataFrame(avg_da_slack_list))

print('\n=== Dangerous Hours (similar dates with rt_slack > 150) ===')
display(pd.DataFrame(dangerous_hours) if dangerous_hours else 'None')


=== 2026-05-30 hr 1 ===
Query dt/hr not in dataset, fetching live...


/tmp/ipykernel_3362251/756036728.py:28: DeprecationWarning: Call to deprecated function get_city_details.
  city_ids    = weather_obj.get_city_details(SPP_CITIES)['CityId'].tolist()


,dt,hr,wind_f (MW),load_f (MW),genoutage_f (MW),avg_temp (°F),henry_gas ($/MMBtu),wind_ramp (MW/hr),load_ramp (MW/hr),wind_ramp_2 (MW/hr),load_ramp_2 (MW/hr)
0,2026-05-30,1,21145.5,35370.0,19284.4,73.0,3.08,417.4,-1232.0,1056.7,-3148.0


RF trained on 55781 rows | top features: [(np.float64(0.2546404443089726), 'avg_temp (°F)'), (np.float64(0.1473164504894639), 'genoutage_f (MW)'), (np.float64(0.13547095455338448), 'henry_gas ($/MMBtu)')]


,rf_proximity,dt,hr,wind_f (MW),load_f (MW),genoutage_f (MW),avg_temp (°F),henry_gas ($/MMBtu),wind_ramp (MW/hr),load_ramp (MW/hr),wind_ramp_2 (MW/hr),load_ramp_2 (MW/hr),SHub_da_slack,SHub_rt_slack,clear_mw,profit_total,profit_congestion,profit_slack
0,1.000,2026-05-30,1,21145.50,35370.0,19284.4,73.000000,3.08,417.40,-1232.0,1056.70,-3148.0,NaN,NaN,NaN,NaN,NaN,NaN
1,0.085,2025-05-13,22,20853.97,35508.0,24553.5,76.500000,3.20,1566.82,-1334.0,2402.55,-2413.0,25.0701,21.9729,161.866999,635.894937,956.971858,-508.317751
2,0.050,2023-10-02,22,21228.98,35677.0,19357.1,77.333333,2.69,1097.94,-1957.0,2367.00,-3311.0,29.9795,18.1064,112.285000,483.600030,573.010788,-331.028958
3,0.045,2025-05-19,21,21439.77,34490.0,22357.5,74.000000,3.07,156.55,-868.0,605.56,-1729.0,30.9905,81.6022,372.362998,-1669.889618,14152.079328,-16638.365290
4,0.040,2025-06-03,20,21845.35,34961.0,15166.1,70.500000,2.98,-1208.81,-1006.0,-1864.07,-1752.0,35.4129,11.0790,291.797999,2562.438037,1516.252697,863.983577


...

,dt,hr,avg_rt_slack
0,2026-05-30,1,21.28
1,2026-05-30,2,20.24
2,2026-05-30,3,19.06
3,2026-05-30,4,23.63
4,2026-05-30,5,24.23
5,2026-05-30,6,21.54
6,2026-05-30,7,33.55
7,2026-05-30,8,19.20
8,2026-05-30,9,26.91
9,2026-05-30,10,65.76



=== Avg DA Slack by Hour ===


,dt,hr,avg_da_slack
0,2026-05-30,1,21.86
1,2026-05-30,2,18.15
2,2026-05-30,3,17.46
3,2026-05-30,4,22.70
4,2026-05-30,5,23.97
5,2026-05-30,6,24.12
6,2026-05-30,7,23.00
7,2026-05-30,8,26.75
8,2026-05-30,9,29.16
9,2026-05-30,10,39.80



=== Dangerous Hours (similar dates with rt_slack > 150) ===


,dt,hr,avg_rt_slack
0,2026-05-30,7,23.0
1,2026-05-30,10,39.8
